In [ ]:

#  dvs_verify.py  -  priprema ulaza + provera/prikaz rezultata za SVE filtre
#  Tok:
#    1) prep_input()   -> napravi sirovi ulaz za mwr + prikaze ulaz
#    2) posle HW obrade i mrd:  verify()  -> ulaz | SW | HW | razlika

import numpy as np
import struct
import matplotlib.pyplot as plt

# Podesavanja
FILTER   = "SOBEL"
RADIUS_SEL = 1     # radijus za BOX (isti kao u main.c)     # BOX | GAUSS | LOG | SHARPEN | SOBEL | CUSTOM
IMG_W    = 128
IMG_H    = 128

SRC_BIN   = "C:/pokusaj_proj/lena_128.bin"     # ulaz sa header-om 
RAW_BIN   = "C:/pokusaj_proj/slika_raw.bin"    # sirov ulaz za mwr (bez header-a)
RESULT_BIN= "C:/pokusaj_proj/result.bin"       # HW izlaz (mrd, sirov)

#  tabela filtara (kao u C-u)
#  svaki: (matrica, radius, norm, scale, mode)   scale < 8, coef/scale u [-1,1)
def get_filter(name):
    if name == "BOX":
        R = RADIUS_SEL; S = 2*R+1
        m = np.ones((S,S)); return m, R, float(S*S), 1.0, 0
    if name == "GAUSS":
        m = np.array([
            [0.0000, 0.0002, 0.0011, 0.0018, 0.0011, 0.0002, 0.0000],
            [0.0002, 0.0029, 0.0131, 0.0216, 0.0131, 0.0029, 0.0002],
            [0.0011, 0.0131, 0.0586, 0.0966, 0.0586, 0.0131, 0.0011],
            [0.0018, 0.0216, 0.0966, 0.1592, 0.0966, 0.0216, 0.0018],
            [0.0011, 0.0131, 0.0586, 0.0966, 0.0586, 0.0131, 0.0011],
            [0.0002, 0.0029, 0.0131, 0.0216, 0.0131, 0.0029, 0.0002],
            [0.0000, 0.0002, 0.0011, 0.0018, 0.0011, 0.0002, 0.0000],
        ]); return m, 3, 1.0, 1.0, 0
    if name == "LOG":
        m = np.array([
            [0.0003, 0.0026, 0.0086, 0.0124, 0.0086, 0.0026, 0.0003],
            [0.0026, 0.0175, 0.0392, 0.0431, 0.0392, 0.0175, 0.0026],
            [0.0086, 0.0392, -0.0000, -0.0965, -0.0000, 0.0392, 0.0086],
            [0.0124, 0.0431, -0.0965, -0.3183, -0.0965, 0.0431, 0.0124],
            [0.0086, 0.0392, -0.0000, -0.0965, -0.0000, 0.0392, 0.0086],
            [0.0026, 0.0175, 0.0392, 0.0431, 0.0392, 0.0175, 0.0026],
            [0.0003, 0.0026, 0.0086, 0.0124, 0.0086, 0.0026, 0.0003],
        ]); return m, 3, 1.0, 1.0, 1
           
    if name == "SHARPEN":
        R = RADIUS_SEL; S = 2*R+1; nb = S*S
        m = -np.ones((S,S))/nb
        m[R,R] += 2.0
        return m, R, 1.0, 3.0, 0
    if name == "SOBEL":
        m = np.array([[-1,0,1],[-2,0,2],[-1,0,1]], float); return m, 1, 1.0, 4.0, 1
    if name == "LAPL4":
        m = np.array([[0,-1,0],[-1,4,-1],[0,-1,0]], float); return m, 1, 1.0, 5.0, 1
    if name == "CUSTOM":
        m = np.array([[0,0,0],[0,1,0],[0,0,0]], float); return m, 1, 1.0, 1.0, 0
    raise ValueError("nepoznat filtar: " + name)

# I/O (header)
def read_bin_img(fn):
    with open(fn, "rb") as f:
        h = struct.unpack("<H", f.read(2))[0]
        w = struct.unpack("<H", f.read(2))[0]
        planes = struct.unpack("<H", f.read(2))[0]
        psz = struct.unpack("<H", f.read(2))[0]
        dt = {1: np.uint8, 2: np.uint16, 4: np.uint32}[psz]
        data = np.frombuffer(f.read(), dtype=dt)
    return data[:h*w].reshape(h, w)

# Fixed-point ref
def sw_reference(inp, m, R, norm, scale, mode):
    """Ista aritmetika kao HW/C: coef=round(m/norm/scale*2^15), sf=round(scale*2^12),
       acc=sum(px*coef), full=acc*sf, mode1: full>>20 (clip s16), mode0: full>>27 (clip u8)."""
    coef = np.round(m / norm / scale * (2**15)).astype(np.int64)
    sf   = int(round(scale * (2**12)))
    H, W = inp.shape
    OUT_H, OUT_W = H - 2*R, W - 2*R
    s = inp.astype(np.int64)
    acc = np.zeros((OUT_H, OUT_W), np.int64)
    for ky in range(-R, R+1):
        for kx in range(-R, R+1):
            acc += s[R+ky:R+ky+OUT_H, R+kx:R+kx+OUT_W] * coef[ky+R, kx+R]
    full = acc * sf
    if mode == 1:
        out = np.clip(full >> 20, -32768, 32767).astype(np.int16)
    else:
        out = np.clip(full >> 27, 0, 255).astype(np.uint8)
    return out


def prep_input():
    """Procita ulaz (sa header-om), snimi sirov za mwr, prikaze ulaz."""
    inp = read_bin_img(SRC_BIN)
    assert inp.shape == (IMG_H, IMG_W), f"dimenzije {inp.shape}, ocekivano {(IMG_H,IMG_W)}"
    inp.astype(np.uint8).tofile(RAW_BIN)
    print(f"Sirov ulaz snimljen: {RAW_BIN}  ({inp.size} B)")
    print(f"U XSCT: mwr -size b -bin -file {RAW_BIN} 0x<DataBuffer> {inp.size}")
    plt.figure(); plt.imshow(inp, cmap="gray"); plt.title("ulaz"); plt.axis("off"); plt.show()
    return inp

def verify():
    """Procita HW rezultat, izracuna SW referencu, prikaze ulaz|SW|HW|razlika."""
    m, R, norm, scale, mode = get_filter(FILTER)
    inp = read_bin_img(SRC_BIN).astype(np.int64)
    OUT_H, OUT_W = IMG_H - 2*R, IMG_W - 2*R

    dt = np.int16 if mode == 1 else np.uint8
    hw = np.fromfile(RESULT_BIN, dtype=dt)
    assert hw.size == OUT_H*OUT_W, f"HW ima {hw.size}, ocekivano {OUT_H*OUT_W} (proveri mrd bajtove/mod)"
    hw = hw.reshape(OUT_H, OUT_W)

    sw = sw_reference(inp, m, R, norm, scale, mode)
    diff = np.abs(hw.astype(np.int64) - sw.astype(np.int64))
    print(f"Filtar: {FILTER}  (R={R}, scale={scale}, mode={mode})")
    print(f"max razlika: {int(diff.max())}   razlicitih: {int((diff!=0).sum())}/{hw.size}")

    fig, ax = plt.subplots(1, 4, figsize=(16, 4))
    ax[0].imshow(inp, cmap="gray");                 ax[0].set_title("ulaz")
    ax[1].imshow(sw,  cmap="gray");                 ax[1].set_title("SOFTVER (ref)")
    ax[2].imshow(hw,  cmap="gray");                 ax[2].set_title("HARDVER")
    ax[3].imshow(diff, cmap="gray");                ax[3].set_title("RAZLIKA")
    for a in ax: a.axis("off")
    plt.tight_layout(); plt.show()


if __name__ == "__main__":
    
    prep_input()
    

In [ ]:
import numpy as np

FILT  = "GAUSS"   # "GAUSS" ili "LOG"
R     = 3         # Izbor radijusa
sigma = 1.0

d = np.arange(-R, R+1); X, Y = np.meshgrid(d, d); r2 = X**2 + Y**2
if FILT == "GAUSS":
    m = np.exp(-r2/(2*sigma**2)); m /= m.sum()      # Normalizovan
    scale, mode = 1.0, 0
else:  # LOG
    m = -1/(np.pi*sigma**4)*(1 - r2/(2*sigma**2))*np.exp(-r2/(2*sigma**2))
    scale, mode = 1.0, 1
print("max|coef| =", round(float(np.abs(m).max()),4), " (mora <", scale, ")   scale=", scale, " mode=", mode)

print("\n--- za C (nalepi u odgovarajuci case, m[%d]) ---" % (m.size))
for row in m:
    print("        " + " ".join(f"{v:+.4f}f," for v in row))

print("\n--- za Python (get_filter) ---")
print("        m = np.array([")
for row in m:
    print("            [" + ", ".join(f"{v:.4f}" for v in row) + "],")
print("        ]); return m, %d, 1.0, %.1f, %d" % (R, scale, mode))

In [ ]:
verify()
